## tl;dr

실제 테스트 DB 11,131종은 공식 AviList v2025b와 내용이 일치했다. NE는 808종(7.259%)이며 전부 BirdLife 링크가 비어 있다. 이 notebook은 저장된 감사 JSON의 결과를 재계산한다. 최신 IUCN 평가를 독립 검증하거나 DB를 재조회하지 않는다.

## Context & Methods

대상: 활성 `rg:concept-set:avilist-v2025b`, `v2025b`, 조류 species. 원 감사는 `scripts/audit_conservation_quality.py`를 통한 실제 테스트 DB 읽기 전용 전수 대조다.

### Key Assumptions

JSON은 저장된 감사 결과이고 DB의 현재 상태를 대변하지 않는다. 비율의 분모는 rows이며 source_mismatches는 원자료 공백 제거 후 비교 결과다.

공식 JSON: https://explore.avilist.org/data/avilist-2025b.json
공식 workbook: https://www.avilist.org/wp-content/uploads/2026/06/AviList-v2025b-10Jun2026-extended.xlsx

원 감사 재현(인증 파일은 산출물에 포함하지 않는다):
```sh
python scripts/audit_conservation_quality.py --snapshot data/avilist-2025b.json --env-file .env.local --concept-set-id rg:concept-set:avilist-v2025b --taxonomy-release v2025b --output docs/verification/assets/2026-10-09-conservation-quality.json
```

별도 실행은 Neo4j 및 프로젝트 의존성과 환경 인증이 필요하다. 이 notebook의 계산은 표준 Python만 사용한다.

## Data

### 1. 저장된 감사 결과 읽기

In [1]:
import json
from pathlib import Path

input_path = Path('2026-10-09-conservation-quality.json')
if not input_path.is_file():
    input_path = Path('docs/verification/assets/2026-10-09-conservation-quality.json')
audit = json.loads(input_path.read_text(encoding='utf-8'))
print(json.dumps({key: audit[key] for key in ['checked_at', 'mode', 'rows', 'expected_species', 'concept_set_id', 'taxonomy_release']}, ensure_ascii=False, indent=2))

{
  "checked_at": "2026-10-09T06:22:41.083872+00:00",
  "mode": "neo4j_read_only",
  "rows": 11131,
  "expected_species": 11131,
  "concept_set_id": "rg:concept-set:avilist-v2025b",
  "taxonomy_release": "v2025b"
}


## Results

### 2. 분모와 NE 비율 재계산

In [2]:
denominator = audit['rows']
ne_count = audit['category_counts']['NE']
assert sum(audit['category_counts'].values()) == denominator
assert ne_count == audit['ne_rows']
assert denominator == audit['expected_species'] == 11131
ne_share = ne_count / denominator
assert abs(ne_share - audit['category_shares']['NE']) < 0.000001
print(f'NE: {ne_count} / {denominator} = {ne_share:.6%}')
print('NE with BirdLife URL:', audit['ne_with_birdlife_url'])

NE: 808 / 11131 = 7.259006%
NE with BirdLife URL: 0


### 3. 차이 목록과 체크섬 검사

In [3]:
zero_lists = ['missing_source_rows', 'missing_graph_sequences', 'source_mismatches', 'invalid_source_identity_or_category']
for key in zero_lists:
    assert audit[key] == [], key
assert audit['duplicate_taxon_ids'] == 0
assert audit['stored_snapshot_checksum_matches'] is True
assert audit['independent_iucn_assessment_check'] is False
assert len(audit['source_format_differences']) == 1
assert audit['ne_with_birdlife_url'] == 0
assert audit['matched_category_without_birdlife_url'] == 0
print('All saved audit invariants passed.')
print('Snapshot SHA256:', audit['comparison_snapshot_sha256'])
print('Formatting difference:', audit['source_format_differences'])

All saved audit invariants passed.
Snapshot SHA256: 3b08845b54b8ab53908aee84d05b0fd8df765e9e01dc655599ca304d9f132411
Formatting difference: [{'taxon_id': 'avilist-taxon:v2025b:17511', 'source_value': 'LC ', 'stored_value': 'LC'}]


### 4. 고정 사례 미리보기

In [4]:
for row in audit['samples']:
    print(row['scientific_name'], row['category_raw'], 'BirdLife link present=' + str(bool(row['birdlife_url'])))

Cyanopsitta spixii EW BirdLife link present=True
Pica serica NE BirdLife link present=False
Pica pica NE BirdLife link present=False
Anas platyrhynchos LC BirdLife link present=True


## Takeaways

적재 정합성은 확인됐다. NE를 실제 미평가 판정으로 단정하지 않고 평가 연결 확인 필요로 전달해야 한다. LC 강제 보정의 근거는 이 감사에 없다. 독립 평가의 종 범위·평가 연도·출처 연결은 후속 검증이 필요하다.

검증 구분: 원 감사는 실제 테스트 DB 조회. 이 notebook은 저장된 JSON 계산 재현. 생물학적 평가 내용은 미검증.

### 구현 회귀 테스트 결과와 감사 재현 범위

최신 origin/main 병합 후 실제 Python 테스트: 전체 642개, 실행 통과 607개, 건너뛰기 35개, 실패 0개. frontend: 253개 통과, 건너뛰기 0개, 실패 0개. 작업 조정자가 확인한 최신 실행 결과이며 이 notebook은 테스트 스위트를 재실행하지 않는다.

실제 DB 전수 감사, 저장된 JSON 수치 재현, 구현 테스트, 독립 IUCN 평가 검증을 서로 구분한다.

테스트 근거: `conservation-python-integrated-test.log`, `conservation-frontend-final-test.log`. 실행 명령의 portable 형식은 `PYTHONPATH=src python -m unittest discover -s tests`, `node --test tests/frontend/*.test.js`다. 머신 경로와 인증정보는 포함하지 않는다.

보고서 HTML QA: 설치된 Chrome에서 환경 불일치로 차트 추출이 실패하여 browser QA를 완료하지 못했다. 최종 패키징은 structural_only이며 light/dark, desktop/narrow, 출처 메뉴/대화상자 확인은 미실행이다.